# check repu (et vs ND)

In [33]:
import pandas as pd
import re

PATH_LISTE_PAYS = "../data/raw/liste_pays_republique_stable.txt"

df = pd.read_csv("../data/interim/2_4_interventions_nettoyees.csv", low_memory=False)

df_brut = pd.read_csv("../data/interim/1_2_extract_15_16_concat.csv", low_memory=False)

df_ND1516 = pd.read_csv(
    "../data/raw/test_regards_citoyens/ND15+16_interventions_hemicycle_rich.tsv",
    sep="\t",
    engine="python",
    on_bad_lines="warn",
)


## Nettoyage du texte :
déjà fait en amont dans notre regroupement mais utile pour autres df


In [34]:
# Trace fonction nettoyage (voir 2_1_filtrage.py)
import unicodedata
import html


def nettoyer_texte(texte):
    if not isinstance(texte, str):
        return ""
    # Normaliser les caractères Unicode
    texte = unicodedata.normalize("NFC", texte)
    # Décoder les entités HTML
    texte = html.unescape(texte)
    # Supprimer les balises HTML/XML > espace (éviter collage de mots)
    texte = re.sub(r"<[^>]+>", " ", texte)
    # Supprimer contenu entre parenthèses
    # NOTE : CHOIX FORT SELON CE QUI VEUT ÊTRE ÉTUDIÉ
    # Supprime des didascalies ("Applaudissements", etc.)
    # mais aussi tout autre contenu entre parenthèses
    # ne gère pas les parenthèses imbriquées mais sont extrêmement rares (parfois sur (e))
    texte = re.sub(r"\([^()]*\)", "", texte)
    # Uniformiser apostrophes (utile pour regex)
    texte = texte.replace("’", "'").replace("\u02bc", "'")
    # Normaliser les espaces (après unescape(), couvre \xa0, \t, \n)
    # et supprimer les espaces multiples
    texte = re.sub(r"\s+", " ", texte).strip()

    return texte


## Regex
Logique de l'identification des mentions valides de République :
- regex sur le champ lexical "républi"
- mais exclusion de certains termes (positions, pas de chaînage) car les
termes exclus peuvent apparaître aussi avec les termes voulus (les idées
républicaines sont menacées par Les Républicains) : chaîner risquerait
de virer des occurrences qu'on aurait voulu garder.

In [35]:
# préparer les pays à exclure
with open(PATH_LISTE_PAYS, "r", encoding="utf-8") as f:
    liste_pays = [line.strip() for line in f]

# pattern regex pour les pays, rendu non capturant plus bas
pattern_pays = r"|".join(re.escape(p) for p in liste_pays)

# Regex de la famille du mot République (simplifié ici)
pattern_lexical = re.compile(
    r"républi",  # même au milieu des mots
    re.I,
)

# Regex des expressions à exclure
# logique : groupes (?:…) non capturant, utilisés juste pour les positions

# Expressions à exclure - casse exacte
# possible cas du féminin… mais pas d'occurrence dans la base avec nos exclusions
pattern_excl_case_sensitive = re.compile(
    # --- Exclusion des occurrences liées au parti les Républicains ---
    r"(?:\b[LlDd]es Républicains\b)"  # garde la casse pour identifier le parti (et pas un adjectif)
    r"|(?:\baux Républicains\b)"  # idem majuscule pour le groupe
    r"|(?:\b[Cc]ollègues? Républicains?\b)"  # cas avec et sans maj pour collègues
    r"|(?:\bsénateurs? Républicains?\b)"  # pas de maj sénateurs ou féminin dans la base après exclu, mais aviser
    r"|(?:\bdéputés? Républicains?\b)"  # pas de maj députés ou féminin dans la base après exclu, mais aviser
    # --- Spécifique corpus AN choisi ---
    r"|(?:\bLes Républicain\b)"  # typo manque s = spécifique corpus AN (4 occurrences)
    r"|(?:\bentre Républicains?\b)"  # spécifique corpus AN (4 occurrences)
    r"|(?:\bex-Républicains?\b)"  # spécifique corpus AN (4 occurrences)
    r"|(?:\banciens? Républicains?\b)"  # spécifique corpus AN (1 occurrence)
    r"|(?:\bseuls Républicains\b)"  # spécifique corpus AN (1 occurrence)
    r"|(?:\bparlementaires Républicains\b)"  # spécifique corpus AN (1 occurrence)
    r"|(?:\bélus Républicains\b)"  # spécifique corpus AN (1 occurrence)
    r"|(?:\bgroupeLes Républicains\b)"  # spécifique corpus AN (1 occurrence)
    r"|(?:\b[Nn]ous Républicains\b)"  # spécifique corpus AN (2 occurrences)
    r"|(?:\bcertains Républicains\b)"  # spécifique corpus AN (4 occurrences)
    r"|(?:\bamis Républicains\b)"  # spécifique corpus AN (1 occurrence)
    # --- Occurrences plusieurs partis ---
    r"|(?:\bdroite, Républicains et macronistes\b)"  # spécifique corpus AN (1 occurrence)
    r"|(?:\bRépublicains-Front national\b)"  # spécifique corpus AN (1 occurrence)
    r"|(?:\bMacronistes, Républicains, lepénistes\b)"  # spécifique corpus AN (1 occurrence)
    r"|(?:\bRassemblement national, Républicains et macronistes\b)"  # spécifique corpus AN (1 occurrence)
    r"|(?:\bparti Républicain\b)"  # spécifique corpus AN (1 occurrence : US)
    # --- Titres de presse ---
    r"|(?:\bL[’']Est républicain\b)"  # le journal
    r"|(?:\bLa Nouvelle République\b)"  # le journal
)

# Expressions à exclure - ignorer la casse
pattern_excl_case_insensitive = re.compile(
    # --- Partis et groupes politiques ---
    # TODO : confirmation MATTHIAS POUR EXCLUSIONS NV CAS PARTIS
    r"(?:\bgauche démocrate et républicaine\b)"  # premier sans |
    r"|(?:\bGauche démocrate républicaine\b)"  # (feinte) ajout léo
    r"|(?:\bGauche démocratique et Républicaine\b)"  # (feinte) ajout léo
    r"|(?:\bgauche démocrate et républicaine-NUPES\b)"
    r"|(?:\bsocialiste, écologiste et républicain\b)"
    r"|(?:\bgroupe socialiste et républicain\b)"  # ajout léo (garder groupe pour limiter flag)
    r"|(?:\bcommuniste républicain citoyen et écologiste\b)"  # ajout léo
    r"|(?:\brépublique en marche\b)"
    r"|(?:\bconstructifs : républicains, UDI, indépendants\b)"  # ajout léo
    r"|(?:\bconstructifs : républicains, UDI et apparentés\b)"  # (feinte) ajout léo
    r"|(?:\bLes Indépendants - République et Territoires\b)"  # ajout léo
    r"|(?:\bLes Indépendants-République et Territoires\b)"  # (feinte) ajout léo
    # TODO : aviser
    # "Rassemblement pour la République" RPR 1 cas -> mais risque appel rassemblement sensible casse ?
    # NOTE : ont également été testés (0 cas ici, mais voir selon autres législatures)
    # "Union des démocrates pour la République" UDR  / "Union des droites pour la République"
    # "Union pour une Nouvelle République" / "Debout la République"
    # "Forum des républicains sociaux" / "Identité et République"
    # --- Fonctions et institutions ---
    # TODO : MATTHIAS CHOISI POUR exclusion présidente(s) de la république
    r"|(?:\bprésidents? de la république\b)"
    r"|(?:\bprésidentes? de la république\b)"  # 7 cas pour féminiser la fonction ou souhaiter élection MLP
    r"|(?:\bprésidences? de la république\b)"
    r"|(?:\bprocureurs? de la république\b)"
    r"|(?:\bcours? de justice de la république\b)"  # nb : cours de sûreté est lui gardé car projet loi LR et pas une institution
    r"|(?:\badministration générale de la république\b)"
    r"|(?:\bgouvernement de la république française\b)"  # pas de pluriel dans corpus
    r"|(?:\bInstitut supérieur des langues de la République française\b)"
    r"|(?:\bHaut-commissariat de la République\b)"  # préfets en Kanaky et Polynésie française uniquement
    r"|(?:\bHaut-commissaire de la République\b)"  # ibid
    r"|(?:\bcompagnies? républicaines? de sécurité\b)"
    r"|(?:\bgarde républicaine\b)"
    r"|(?:\bgardes? républicains?\b)"
    r"|(?:\buniversités? de la République\b)"  # sur corpus 2017-2024 réf à une commission d'enquête
    r"|(?:\binstitut Famille et République\b)"  # institut privé de la galaxie LMPT
    # --- Titres de lois ---
    r"|(?:\bnouvelle organisation territoriale de la République\b)"
    r"|(?:\bconfortant le respect des principes de la République\b)"
    r"|(?:\bpour une république numérique\b)"
    # --- Expression et législations ---
    # NOTE: Le choix a été fait de ne pas exclure ces formes qui sont pertinentes à garder
    # Elles sont conservées ici en commentaires pour traçabilité
    # r"|(?:\bcontrat d[’']engagement républicain\b)"
    # r"|(?:\bcontrat d[’']engagement au respect des principes de la République\b)"
    # r"|(?:\bcontrat d[’']intégration républicaine\b)"
    # r"|(?:\bquartier[s]? de reconquête républicaine\b)"
    # --- Lieux (places, monuments) ---
    r"|(?:\bplace de la République\b)"  # (45 occurrences)
    # --- Pays, territoires, entités, etc. ---
    r"|(?:\brépubliques? soviétiques?\b)"
    r"|(?:\bex-républiques? soviétiques?\b)"
    r"|(?:\brépublique de Weimar\b)"
    r"|(?:\bRépublique yougoslave\b)"  # spécifique corpus AN
    r"|(?:\brépublique du Haut-Karabakh\b)"  # spécifique corpus AN
    r"|(?:\brépublique d[’']Artsakh\b)"  # spécifique corpus AN
    r"|(?:\brépublique de l[’']Artsakh\b)"  # spécifique corpus AN
    r"|(?:\brépublique des Fidji\b)"  # spécifique corpus AN
    r"|(?:\bRépublique de Chine\b)"  # spécifique corpus AN
    r"|(?:\brépubliques? du Donbass\b)"  # spécifique corpus AN
    r"|(?:\brépublique de Crimée\b)"  # spécifique corpus AN
    r"|(?:\bRépublique démocratique d[’']Arménie\b)"  # spécifique corpus AN
    r"|(?:\bRépubliques du Bénin et du Sénégal\b)"  # spécifique corpus AN
    r"|(?:\b(?:" + pattern_pays + r")\b)",  # ajout des exclusions de pays (liste)
    re.I,
)

# TODO / NOTE : quelques (~10) "république islamique" sans précision pour parler de l'Iran
# mais risque de supprimer d'autres occurrences que l'on veut garder,
# ou alors aviser majuscule a République vs sans ? -> trop niche


# Fonction de décompte des occurrences
def count_lexical_outside_excl(text):
    """
    Compte les occurrences valides du champ lexical "républi" (hors zones
    d'exclusion). Early-exit via pattern_lexical.search() avant de calculer
    les positions d'exclusion (coûteux, notamment la liste de pays) : utile
    car la grande majorité des textes ne contiennent aucune occurrence.

    NOTE : l'ancienne fonction contains_lexical_outside_excl() a été
    supprimée (07/07/2026) : elle est strictement équivalente à
    (count_lexical_outside_excl(text) > 0), donc redondante. Équivalence
    vérifiée par test, même nombre de matchs.

    NOTE : on pourrait optimiser in_excl() via spans triés + bisect, pas
    indispensable ici et plus complexe.
    + probablement pas rentable car pas assez occurrences par texte ?
    """
    if pd.isna(text) or not pattern_lexical.search(text):
        return 0
    # Trouver les positions des expressions exclues
    excl_positions = [m.span() for m in pattern_excl_case_sensitive.finditer(text)] + [
        m.span() for m in pattern_excl_case_insensitive.finditer(text)
    ]

    # Fonction pour vérifier si une position est dans une zone exclue
    def in_excl(pos):
        for start, end in excl_positions:
            if start <= pos < end:
                return True
        return False

    return sum(1 for m in pattern_lexical.finditer(text) if not in_excl(m.start()))


## Application

In [36]:
### df regroupé

# ========== Appliquer comptage et identification mentions valides ==========
df["nombre_mentions_repu"] = df["texte"].apply(count_lexical_outside_excl)
df["repu_match_valide"] = df["nombre_mentions_repu"] > 0
print(df["repu_match_valide"].value_counts())
print(df.shape)

# check avec le texte brut du df regroupé au cas ou
print("-----------------")

# ========== Appliquer comptage et identification mentions valides ==========
df["nombre_mentions_repu_texte_brut"] = df["texte_brut"].apply(
    count_lexical_outside_excl
)
df["repu_match_valide_texte_brut"] = df["nombre_mentions_repu_texte_brut"] > 0
print(df["repu_match_valide_texte_brut"].value_counts())
print(df.shape)

repu_match_valide
False    506192
True      10831
Name: count, dtype: int64
(517023, 68)
-----------------
repu_match_valide_texte_brut
False    505725
True      11298
Name: count, dtype: int64
(517023, 70)


In [37]:
### df_brut sans regroupement
print("-----------------")
print("--- df_brut ---")
print("-----------------")

# ========== Appliquer comptage et identification mentions valides ==========
df_brut["nombre_mentions_repu"] = df_brut["texte"].apply(count_lexical_outside_excl)
df_brut["repu_match_valide"] = df_brut["nombre_mentions_repu"] > 0
print(df_brut["repu_match_valide"].value_counts())
print(df_brut.shape)


### df_brut sans regroupement mais nettoyage
print("-----------------")

# ========== Appliquer comptage et identification mentions valides ==========
df_brut["nombre_mentions_repu_net"] = (
    df_brut["texte"].apply(nettoyer_texte).apply(count_lexical_outside_excl)
)
df_brut["repu_match_valide_net"] = df_brut["nombre_mentions_repu_net"] > 0
print(df_brut["repu_match_valide_net"].value_counts())
print(df_brut.shape)

-----------------
--- df_brut ---
-----------------
repu_match_valide
False    1114011
True       13818
Name: count, dtype: int64
(1127829, 39)
-----------------
repu_match_valide_net
False    1114734
True       13095
Name: count, dtype: int64
(1127829, 41)


In [38]:
### df_ND1516

print("-----------------")
print("--- df_ND1516 ---")
print("-----------------")


# ========== Appliquer comptage et identification mentions valides ==========
df_ND1516["nombre_mentions_repu"] = df_ND1516["intervention"].apply(
    count_lexical_outside_excl
)
df_ND1516["repu_match_valide"] = df_ND1516["nombre_mentions_repu"] > 0
print(df_ND1516["repu_match_valide"].value_counts())
print(df_ND1516.shape)


### df_ND1516 mais avec nettoyage
print("-----------------")

# ========== Appliquer comptage et identification mentions valides ==========
df_ND1516["nombre_mentions_repu_net"] = (
    df_ND1516["intervention"].apply(nettoyer_texte).apply(count_lexical_outside_excl)
)
df_ND1516["repu_match_valide_net"] = df_ND1516["nombre_mentions_repu_net"] > 0
print(df_ND1516["repu_match_valide_net"].value_counts())
print(df_ND1516.shape)


-----------------
--- df_ND1516 ---
-----------------
repu_match_valide
False    1377532
True       13675
Name: count, dtype: int64
(1391207, 18)
-----------------
repu_match_valide_net
False    1377532
True       13675
Name: count, dtype: int64
(1391207, 20)


#### Test sans les cas intervenants manquants

In [39]:
# Cas extraction brute
mask_extract_speaker = (
    df_brut["id_acteur"].notna()
    | df_brut["id_orateur"].notna()
    | df_brut["nom_orateur"].fillna("").astype(str).str.strip().ne("")
)
df_brut_with_speaker = df_brut[mask_extract_speaker].copy()

print("-----------------")
print("--- df_brut_with_speaker ---")
print("-----------------")
print("df_brut_with_speaker shape : ", df_brut_with_speaker.shape)
print("-----------------")
print(
    "df_brut_with_speaker repu_match_valide counts :\n",
    df_brut_with_speaker["repu_match_valide"].value_counts(),
)
print("-----------------")
print(
    "df_brut_with_speaker repu_match_valide_net counts :\n",
    df_brut_with_speaker["repu_match_valide_net"].value_counts(),
)

# cas nd
mask_ND_speaker = df_ND1516["parlementaire"].fillna("").astype(str).str.strip().ne(
    ""
) | df_ND1516["personnalite"].fillna("").astype(str).str.strip().ne("")
df_ND1516_with_speaker = df_ND1516[mask_ND_speaker].copy()
print("-----------------")
print("--- df_ND1516_with_speaker ---")
print("-----------------")
print("df_ND1516_with_speaker shape : ", df_ND1516_with_speaker.shape)
print("-----------------")
print(
    "df_ND1516_with_speaker repu_match_valide counts :\n",
    df_ND1516_with_speaker["repu_match_valide"].value_counts(),
)
print("-----------------")
print(
    "df_ND1516_with_speaker repu_match_valide_net counts :\n",
    df_ND1516_with_speaker["repu_match_valide_net"].value_counts(),
)


-----------------
--- df_brut_with_speaker ---
-----------------
df_brut_with_speaker shape :  (1027685, 41)
-----------------
df_brut_with_speaker repu_match_valide counts :
 repu_match_valide
False    1013868
True       13817
Name: count, dtype: int64
-----------------
df_brut_with_speaker repu_match_valide_net counts :
 repu_match_valide_net
False    1014590
True       13095
Name: count, dtype: int64
-----------------
--- df_ND1516_with_speaker ---
-----------------
df_ND1516_with_speaker shape :  (1088105, 20)
-----------------
df_ND1516_with_speaker repu_match_valide counts :
 repu_match_valide
False    1074482
True       13623
Name: count, dtype: int64
-----------------
df_ND1516_with_speaker repu_match_valide_net counts :
 repu_match_valide_net
False    1074482
True       13623
Name: count, dtype: int64


## Confrontation écart des cas vs ND

### Par confrontation pnum / id_syceron

In [40]:
# NOTE : pourrait optimiser calcul en réutilisant étapes précédentes
# mais j'ai séparé ici pour modularité en mode tests séparés
import pandas as pd
import re

P_NUMBER_RE = re.compile(r"#P?(\d+)", re.I)


def extract_pnum(url):
    if not url or pd.isna(url):
        return None
    url = str(url).strip()
    m = P_NUMBER_RE.search(url)
    return m.group(1) if m else None


# --- Chargement ---
df_brut_comp = pd.read_csv(
    "../data/interim/1_2_extract_15_16_concat.csv", low_memory=False
)
df_nd_comp = pd.read_csv(
    "../data/raw/test_regards_citoyens/ND15+16_interventions_hemicycle_rich.tsv",
    sep="\t",
    engine="python",
    on_bad_lines="warn",
)

# --- Normaliser les deux corpus (une seule fois, réutilisé pour tout le reste) ---
df_nd_comp["interv_net"] = (
    df_nd_comp.get("intervention", "").fillna("").apply(nettoyer_texte)
)
df_brut_comp["texte_net"] = (
    df_brut_comp.get("texte", "").fillna("").apply(nettoyer_texte)
)

df_nd_comp["nb_repu"] = df_nd_comp["interv_net"].apply(count_lexical_outside_excl)
df_nd_comp["match_nd"] = df_nd_comp["nb_repu"] > 0

df_brut_comp["nb_repu"] = df_brut_comp["texte_net"].apply(count_lexical_outside_excl)
df_brut_comp["match_brut"] = df_brut_comp["nb_repu"] > 0

# --- Comparaison pnum vs id_syceron ---

# Colonne pnum
df_nd_comp["pnum"] = pd.to_numeric(
    df_nd_comp["source"].apply(extract_pnum), errors="raise"
).astype("Int64")

# --- Harmoniser les clés ---
df_brut_comp["id_syceron"] = df_brut_comp["id_syceron"].astype("Int64")
df_nd_comp["pnum"] = df_nd_comp["pnum"].astype("Int64")

# --- Vérifier l'unicité des clés avant de conclure quoi que ce soit ---
print("Doublons id_syceron :", df_brut_comp["id_syceron"].duplicated().sum())
print("Doublons pnum :", df_nd_comp["pnum"].duplicated().sum())

# --- Isoler les clés ND absentes de l'extraction brute ---
cles_brut = set(df_brut_comp["id_syceron"])
df_nd_only = df_nd_comp[~df_nd_comp["pnum"].isin(cles_brut)]

print("Lignes ND absentes de l'extraction brute :", df_nd_only.shape[0])

# --- Restreindre aux cas qui nous intéressent : ND a un match "républi" ---
df_nd_only_match = df_nd_only[df_nd_only["match_nd"]]
print("Dont avec mention valide de République :", df_nd_only_match.shape[0])

# --- Export pour inspection ---
# df_nd_only.to_csv("nd_absentes_de_brut.csv", index=False)
df_nd_only_match.to_csv("nd_pnum_absentes_de_brut_match.csv", index=False)

Doublons id_syceron : 367
Doublons pnum : 281705
Lignes ND absentes de l'extraction brute : 45410
Dont avec mention valide de République : 130


### Par  check snippet texte

In [41]:
# NOTE : repart ici de ce qui est fait dans comp pnum id_syceron

# --- Ne garder QUE les lignes pertinentes pour match "républi"+exclusions (gain de perf) ---
df_nd_repu = df_nd_comp[df_nd_comp["nb_repu"] > 0].copy()
df_brut_repu = df_brut_comp[df_brut_comp["nb_repu"] > 0].copy()

print("Lignes ND avec mention valide   :", df_nd_repu.shape[0])
print("Lignes brut avec mention valide :", df_brut_repu.shape[0])

# --- Construire le "big blob" seulement à partir des lignes concernées ---
# (plus rapide que sur tout le corpus, et suffisant pour la recherche de snippet)
big_brut_repu = " ".join(df_brut_repu["texte_net"].tolist())
big_nd_repu = " ".join(df_nd_repu["interv_net"].tolist())


def search_snippets_fast(source_df, source_text_col, big_target, snippet_len=150):
    df_out = source_df.copy()
    df_out["snippet"] = df_out[source_text_col].fillna("").str[:snippet_len]
    df_out["found"] = df_out["snippet"].apply(
        lambda s: bool(s) and (s in big_target)
    )
    return df_out

# --- Cas qui qui nous intéressent : ND mentionne "républi", cherche si ça existe aussi côté brut ---
res_nd_in_brut = search_snippets_fast(
    df_nd_repu, "interv_net", big_brut_repu, snippet_len=150
)

# --- Les vrais absents : présents dans ND, mention "républi", introuvables dans brut ---
nd_absent_de_brut = res_nd_in_brut[~res_nd_in_brut["found"]]
print(
    "ND avec 'républi' introuvable dans brut (snippets) :",
    nd_absent_de_brut.shape[0],
    "/",
    len(res_nd_in_brut),
)

nd_absent_de_brut.to_csv("nd_snippets_absent_de_brut.csv", index=False)

# --- Ceux qui sont dans brut mais pas dans ND ---
res_brut_in_nd = search_snippets_fast(
    df_brut_repu, "texte_net", big_nd_repu, snippet_len=150
)
brut_absent_de_nd = res_brut_in_nd[~res_brut_in_nd["found"]]
print(
    "Brut avec 'républi' introuvable dans ND :",
    brut_absent_de_nd.shape[0],
    "/",
    len(res_brut_in_nd),
)
brut_absent_de_nd.to_csv("brut_snippets_absent_de_nd.csv", index=False)


Lignes ND avec mention valide   : 13675
Lignes brut avec mention valide : 13095
ND avec 'républi' introuvable dans brut (snippets) : 1229 / 13675
Brut avec 'républi' introuvable dans ND : 2577 / 13095


In [42]:
brut_absent_de_nd.to_csv("brut_snippets_absent_de_nd.csv", index=False)


# tests en vrac

In [43]:
texte = "…pour inscrire dans le droit commun des mesures efficaces assurant la protection de nos concitoyens.<br/>N’oublions pas ceux qui perdent la vie dans ces actes terroristes ! N’oublions pas que les victimes du terrorisme sont notre démocratie et notre innocence ! N’oublions pas que nous devons la sécurité à nos concitoyens, que nous, élus de la République, sommes responsables ! Je vois un pays splendide et un peuple de génie résister à cet abîme ; je vois les êtres pour lesquels nous nous sommes engagés, apaisés, déterminés et libres. Ce que nous allons accomplir aujourd’hui est bien meilleur que tous nos actes passés. <italique>(Applaudissements sur les bancs du groupe REM.)</italique></texte>"


In [44]:
nettoyer_texte(texte)

"…pour inscrire dans le droit commun des mesures efficaces assurant la protection de nos concitoyens. N'oublions pas ceux qui perdent la vie dans ces actes terroristes ! N'oublions pas que les victimes du terrorisme sont notre démocratie et notre innocence ! N'oublions pas que nous devons la sécurité à nos concitoyens, que nous, élus de la République, sommes responsables ! Je vois un pays splendide et un peuple de génie résister à cet abîme ; je vois les êtres pour lesquels nous nous sommes engagés, apaisés, déterminés et libres. Ce que nous allons accomplir aujourd'hui est bien meilleur que tous nos actes passés."